#Negative BerTopic

In [ ]:
%pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.9 MB/s eta 0:00:00


In [ ]:
# Imports
from bertopic import BERTopic
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = "/content/drive/MyDrive/Twitter/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_relevant_neg.csv"
file = pd.read_csv(path)
print(file.head())

   Unnamed: 0                                               Text  \
0           2  That’s so untrue. I’ve done both human and AI ...   
1           3  The worst thing that OpenAI used Wikipedia and...   
2           7  Paid a hefty price for a therapist that was no...   
3           9  Outsourcing model computational power to local...   
4          15  I think i have lost alot of Friends and it wor...   

                                           sentiment  
0  {'label': 'negative', 'score': 0.5481274127960...  
1  {'label': 'negative', 'score': 0.9413481950759...  
2  {'label': 'negative', 'score': 0.8396270275115...  
3  {'label': 'negative', 'score': 0.6113526225090...  
4  {'label': 'negative', 'score': 0.8383101224899...  


In [ ]:
keywords = ['chatgpt', 'bot', 'ai', 'mental']
# keywords = ['chatgpt', 'chat', 'gpt', 'ai', 'therapy', 'therapist', 'therapists', 'mental', 'health']
import re

def clean_youtube(file, keywords):
  docs = []
  for string in file["Text"]:
      text = str(string).lower()

      for k in keywords:
        #text = text.replace(k, "")
        text = re.sub(r'\b' + k + r'\b', '', text)

      text = re.sub(r"http\S+|www\.\S+|\b\w+\.(?:ly|com|org|net|io)\S*", " ", text)
      text = re.sub(r"[^\x20-\x7E]", " ", text)
      text = re.sub(r"\b\d+\b", " ", text)
      text = re.sub(r"[|^{}\[\]<>~`@=#]", " ", text)
      text = re.sub(r"\s+", " ", text).strip()

      docs.append(text)

  # Remove exact duplicates
  docs = sorted(list(set(docs)))

  # Remove near-duplicates
  seen = set()
  unique_docs = []
  for d in docs:
      words = d.split()
      tail = ' '.join(words[len(words)//5:])
      if tail not in seen:
          seen.add(tail)
          unique_docs.append(d)
  docs = unique_docs

  # Remove short comments
  docs = [d for d in docs if len(d.split()) >= 8]

  return docs


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

custom_stops = list(CountVectorizer(stop_words="english").get_stop_words())
custom_stops += ['like', 'just', 'don', 'use', 'know', 'really',
                 'want', 'think', 've', 'good', 'going', 'new']
custom_stops += ['bot', 'chatbot', 'ai', 'therapist', 'therapists', 'therapy',
                 'gpt', 'mental', 'health', 'person', 'people', 'human', 'humans']
custom_stops += ['need', 'actually', 'real', 'using']

def run_youtube(file, keywords, n_topics=8):
    vectorizer_model = CountVectorizer(stop_words=custom_stops, min_df=2)
    cluster_model = KMeans(n_clusters=n_topics, random_state=42)
    topic_model = BERTopic(
        embedding_model="all-MiniLM-L6-v2",
        vectorizer_model=vectorizer_model,
        hdbscan_model=cluster_model
    )
    docs = clean_youtube(file, keywords)

    topics, prob = topic_model.fit_transform(docs)
    #topic_model.reduce_topics(docs, nr_topics=n_topics)
    topics = topic_model.topics_

    return topics, prob, topic_model.get_topic_info(), topic_model, docs

In [ ]:
# running BerTopic
from google.colab import sheets

n_topics, n_prob, n_topic_info, n_topic_model, n_docs = run_youtube(file, keywords, n_topics=5)
sheet = sheets.InteractiveSheet(df=n_topic_info)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


https://docs.google.com/spreadsheets/d/1Yvkj-R_NxNH3QiV0Zq4ejPGSWTFwRrIPnXQ7j5FH29Y/edit#gid=0


# Validation for Negative BerTopic

In [ ]:
# Silhoutte
# This checks if documents are in correct groups

In [ ]:
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(n_docs)

score = silhouette_score(embeddings, n_topic_model.topics_)
print(f"Silhouette Score: {score}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Silhouette Score: 0.030089257284998894


In [ ]:
# Topic coherence
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 32.3 MB/s eta 0:00:00


In [ ]:
# Coherence is not effective for BerTopic

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

# Get topic words from BERTopic
topic_words = [[word for word, _ in n_topic_model.get_topic(t)] for t in n_topic_model.get_topics() if t != -1]

# Tokenize the cleaned docs
tokenized_docs = [doc.split() for doc in n_docs]

# Calculate coherence
dictionary = Dictionary(tokenized_docs)
coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)

print(f"BERTopic Coherence Score: {coherence_model.get_coherence()}")

BERTopic Coherence Score: 0.36079361103873014


In [ ]:
def run_bertDiversity(r_topic_model):
  unique_words = set()
  all_words = []
  for t in r_topic_model.get_topics():
      if t != -1:
          words = [w for w, _ in r_topic_model.get_topic(t)]
          all_words.extend(words)
          unique_words.update(words)
  print(f"Topic Diversity: {len(unique_words) / len(all_words)}")
run_bertDiversity(n_topic_model)

Topic Diversity: 0.8


In [ ]:
def run_bertRandomSeed(docs, n_topics = 5):
  seeds = [42, 123, 456]
  for seed in seeds:
      cluster_model = KMeans(n_clusters=n_topics, random_state=seed)
      topic_model = BERTopic(
          embedding_model="all-MiniLM-L6-v2",
          vectorizer_model=CountVectorizer(stop_words="english", min_df=2),
          hdbscan_model=cluster_model
      )
      topics, prob = topic_model.fit_transform(docs)
      print(f"\nSeed {seed}:")
      for t in topic_model.get_topics():
          if t != -1:
              words = [w for w, _ in topic_model.get_topic(t)][:5]
              print(f"  Topic {t}: {words}")
run_bertRandomSeed(n_docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 42:
  Topic 0: ['people', 'just', 'like', 'use', 'human']
  Topic 1: ['therapy', 'therapist', 'like', 'therapists', 'just']
  Topic 2: ['chat', 'gpt', 'just', 'therapist', 'people']
  Topic 3: ['therapy', 'therapists', 'therapist', 'human', 'people']
  Topic 4: ['just', 'people', 'like', 'friends', 'feel']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 123:
  Topic 0: ['therapy', 'therapist', 'therapists', 'like', 'people']
  Topic 1: ['people', 'just', 'like', 'use', 'human']
  Topic 2: ['just', 'friends', 'like', 'people', 'talk']
  Topic 3: ['people', 'like', 'chat', 'human', 'just']
  Topic 4: ['chat', 'gpt', 'better', 'therapist', 'like']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 456:
  Topic 0: ['therapy', 'therapist', 'therapists', 'people', 'like']
  Topic 1: ['people', 'just', 'like', 'use', 'human']
  Topic 2: ['just', 'like', 'talk', 'know', 'friends']
  Topic 3: ['people', 'chat', 'like', 'human', 'chatbots']
  Topic 4: ['chat', 'gpt', 'better', 'therapist', 'like']


# NMF Negative

In [ ]:
# imports
import numpy as np
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk import pos_tag

In [ ]:
def negative_nouns_adj(text):
    is_noun_adj = lambda pos: pos[:2] == 'NN' or pos[:2] == 'JJ'
    tokenized = word_tokenize(text)
    return ' '.join([word for word, pos in pos_tag(tokenized) if is_noun_adj(pos)])

def neg_clean_data_nmf_yt(file, keywords):
  docs = []
  for string in file["Text"]:
    text = str(string).lower()

    for k in keywords:
        text = text.replace(k, "")

    text = re.sub(r"http\S+|www\.\S+|\b\w+\.(?:ly|com|org|net|io)\S*", " ", text)
    text = re.sub(r"[^\x20-\x7E]", " ", text)
    text = re.sub(r"\b\d+\b", " ", text)
    text = re.sub(r"[|^{}\[\]<>~`@=#]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    string = negative_nouns_adj(text)
    docs.append(string)

  docs = list(set(docs))
  docs = [d for d in docs if len(d.split()) >= 5]

  return docs

In [ ]:
def run_nmf_yt(file, keys, numTop):
    docs = neg_clean_data_nmf_yt(file, keys)

    custom_stops = list(TfidfVectorizer(stop_words="english").get_stop_words())
    custom_stops += ['like', 'just', 'don', 'use', 'know', 'really',
                 'want', 'think', 've', 'good', 'going', 'new',
                 'things', 'better', 'need', 'way', 'using', 'help',
                 'does', 'right', 'lot', 'time', 'say', 'feel',
                 'make', 'actually', 'tell', 'chat',
                 'ai', 'chatgpt', 'gpt', 'therapist', 'therapists',
                 'therapy', 'mental', 'health', 'human', 'humans',
                 'people', 'person', 'bot', 'chatbot']

    vectorizer = TfidfVectorizer(
        stop_words='english',
        max_df=0.8,
        min_df=2
    )
    X = vectorizer.fit_transform(docs)

    nmf = NMF(n_components=numTop, random_state=42)
    W = nmf.fit_transform(X)
    H = nmf.components_

    topic_words = []

    vocab = vectorizer.get_feature_names_out()
    for k in range(numTop):
        top_idx = H[k].argsort()[::-1][:10]
        words = [vocab[i] for i in top_idx]
        topic_words.append(words)
    doc_topics = W.argmax(axis=1)
    return doc_topics, topic_words, docs

In [ ]:
file = pd.read_csv(path)

keys = ['chatgpt', 'chat gpt', 'gpt', 'ai', 'therapist', 'therapists',
           'therapy', 'mental', 'health', 'human', 'humans']
doc_topics, topic_words, docs = run_nmf_yt(file, keys, 4)
for i, words in enumerate(topic_words):
    print(f"Topic {i}: {words}")

Topic 0: ['real', 'good', 'better', 'advice', 'person', 'thing', 'problem', 'sd', 'life', 'help']
Topic 1: ['people', 'shit', 'problem', 'self', 'psychosis', 'social', 'choice', 'free', 'comments', 'money']
Topic 2: ['time', 'way', 'things', 'wrong', 'issues', 'times', 'lot', 'questions', 'long', 'problems']
Topic 3: ['bad', 'idea', 'right', 'dont', 'video', 'fake', 'school', 'person', 'fault', 'friends']


In [ ]:
tokenized_docs = [doc.split() for doc in docs]
dictionary = Dictionary(tokenized_docs)

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)

print(f"NMF Coherence Score: {coherence_model.get_coherence()}")

# NMF seems to score lower on this data set compared to LDA

NMF Coherence Score: 0.4018891473348538


# Separate Youtube Positive and Neutral Comments

In [ ]:
path2 = "/content/drive/MyDrive/Twitter/Social Media Analysis of ChatGPT for Therapy/Collected Data/yt_relevant_neu_pos.csv"

In [ ]:
import ast

file = pd.read_csv(path2)

# Parse the sentiment string into actual dict
file['sentiment_parsed'] = file['sentiment'].apply(ast.literal_eval)
file['label'] = file['sentiment_parsed'].apply(lambda x: x['label'])

# Split into two dataframes
neutral_df = file[file['label'] == 'neutral'].reset_index(drop=True)
positive_df = file[file['label'] == 'positive'].reset_index(drop=True)

#print(f"Neutral: {len(neutral_df)}")
#print(f"Positive: {len(positive_df)}")
#positive_df.head()
neutral_df.head()

,Unnamed: 0,Text,sentiment,sentiment_parsed,label
0,0,"Well appreciate your ""facts"" for sure but rega...","{'label': 'neutral', 'score': 0.6064373254776001}","{'label': 'neutral', 'score': 0.6064373254776001}",neutral
1,6,ChatGPT is my therapist but I m older,"{'label': 'neutral', 'score': 0.7344046235084534}","{'label': 'neutral', 'score': 0.7344046235084534}",neutral
2,11,This is probably not relevant at all but I got...,"{'label': 'neutral', 'score': 0.6554722785949707}","{'label': 'neutral', 'score': 0.6554722785949707}",neutral
3,12,"Personally, I think people are mistaken when t...","{'label': 'neutral', 'score': 0.5861011743545532}","{'label': 'neutral', 'score': 0.5861011743545532}",neutral
4,13,I just have my ChatGPT tell me bible verses an...,"{'label': 'neutral', 'score': 0.717455267906189}","{'label': 'neutral', 'score': 0.717455267906189}",neutral


# BerTopic Positive Comments

In [ ]:
pos_topics, pos_prob, pos_topic_info, pos_topic_model, pos_docs = run_youtube(positive_df, keywords, n_topics=5)
sheet = sheets.InteractiveSheet(df=pos_topic_info)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


https://docs.google.com/spreadsheets/d/1iwci5w1Os8ZvlYFgwPB9sPo--ANcY09qvcmDk8WucTI/edit#gid=0


In [ ]:
# Verification Part
# Silhouette
embeddings = model.encode(pos_docs)

score = silhouette_score(embeddings, pos_topic_model.topics_)
print(f"Silhouette Score: {score}")

Silhouette Score: 0.022444911301136017


In [ ]:
# Coherence
# Get topic words from BERTopic
topic_words = [[word for word, _ in pos_topic_model.get_topic(t)] for t in pos_topic_model.get_topics() if t != -1]

# Tokenize the cleaned docs
tokenized_docs = [doc.split() for doc in pos_docs]

# Calculate coherence
dictionary = Dictionary(tokenized_docs)
coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)

print(f"BERTopic Coherence Score: {coherence_model.get_coherence()}")

BERTopic Coherence Score: 0.26420617660570117


# NMF Positive Youtube Comments

In [ ]:
doc_topics, topic_words, docs = run_nmf_yt(positive_df, keys, 4)
for i, words in enumerate(topic_words):
    print(f"Topic {i}: {words}")

Topic 0: ['people', 'useful', 'problems', 'video', 'thing', 'experience', 'help', 'time', 'real', 'lot']
Topic 1: ['good', 'advice', 'thing', 'lot', 'question', 'information', 'holy', 'bad', 'response', 'world']
Topic 2: ['better', 'best', 'free', 'gemini', 'friend', 'new', 'results', 'information', 'ones', 'friends']
Topic 3: ['great', 'person', 'helpful', 'way', 'life', 'tool', 'thoughts', 'real', 'able', 'time']


In [ ]:
# Validation
tokenized_docs = [doc.split() for doc in docs]
dictionary = Dictionary(tokenized_docs)

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)

print(f"NMF Coherence Score: {coherence_model.get_coherence()}")

NMF Coherence Score: 0.44440310043132236


# BerTopic Neutral Comments

In [ ]:
neu_topics, neu_prob, neu_topic_info, neu_topic_model, neu_docs = run_youtube(neutral_df, keywords, n_topics=5)
sheet = sheets.InteractiveSheet(df=neu_topic_info)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


https://docs.google.com/spreadsheets/d/1I_SE_lV3OzvpCyNdbeKKp65QS-IaXGBvx8WR66WV0xc/edit#gid=0


In [ ]:
# Validation
topic_words = [[word for word, _ in neu_topic_model.get_topic(t)] for t in neu_topic_model.get_topics() if t != -1]
tokenized_docs = [doc.split() for doc in neu_docs]
dictionary = Dictionary(tokenized_docs)

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)

print(f"Neutral BERTopic Coherence: {coherence_model.get_coherence()}")

Neutral BERTopic Coherence: 0.2472149776795891


In [ ]:
#Topic diversity
unique_words = set()
all_words = []
for t in neu_topic_model.get_topics():
    if t != -1:
        words = [w for w, _ in neu_topic_model.get_topic(t)]
        all_words.extend(words)
        unique_words.update(words)

diversity = len(unique_words) / len(all_words)
print(f"Topic Diversity: {diversity}")
# Appears to be a large amount of topic diversity

Topic Diversity: 0.8


In [ ]:
# results demonstrate that they come from patterns and are not random
seeds = [42, 123, 456]
for seed in seeds:
    cluster_model = KMeans(n_clusters=5, random_state=seed)
    topic_model = BERTopic(
        embedding_model="all-MiniLM-L6-v2",
        vectorizer_model=CountVectorizer(stop_words=custom_stops, min_df=2),
        hdbscan_model=cluster_model
    )
    topics, prob = topic_model.fit_transform(neu_docs)
    print(f"\nSeed {seed}:")
    for t in topic_model.get_topics():
        if t != -1:
            words = [w for w, _ in topic_model.get_topic(t)][:5]
            print(f"  Topic {t}: {words}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 42:
  Topic 0: ['help', 'feel', 'way', 'need', 'time']
  Topic 1: ['video', 'help', 'need', 'replace', 'isn']
  Topic 2: ['ask', 'try', 'tell', 'prompt', 'feel']
  Topic 3: ['chat', 'game', 'using', 'better', 'way']
  Topic 4: ['real', 'technology', 'chat', 'does', 'language']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 123:
  Topic 0: ['help', 'feel', 'way', 'need', 'time']
  Topic 1: ['video', 'support', 'need', 'help', 'woebot']
  Topic 2: ['ask', 'prompt', 'try', 'tell', 'said']
  Topic 3: ['chat', 'game', 'using', 'better', 'way']
  Topic 4: ['real', 'technology', 'chat', 'does', 'language']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Seed 456:
  Topic 0: ['help', 'feel', 'way', 'need', 'time']
  Topic 1: ['replace', 'help', 'need', 'provide', 'cbt']
  Topic 2: ['ask', 'tell', 'way', 'prompt', 'try']
  Topic 3: ['chat', 'game', 'using', 'way', 'better']
  Topic 4: ['technology', 'real', 'bots', 'video', 'language']


In [ ]:
# Silhouette
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(neu_docs)

score = silhouette_score(embeddings, neu_topic_model.topics_)
print(f"Silhouette Score: {score}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Silhouette Score: 0.026030467823147774


# NMF Neutral Comments

In [ ]:
doc_topics, topic_words, docs = run_nmf_yt(neutral_df, keys, 4)
for i, words in enumerate(topic_words):
    print(f"Topic {i}: {words}")

Topic 0: ['people', 'way', 'good', 'things', 'lot', 'advice', 'time', 'thing', 'problem', 'person']
Topic 1: ['real', 'person', 'life', 'god', 'time', 'bots', 'world', 'companies', 'local', 'voice']
Topic 2: ['questions', 'answers', 'question', 'thoughts', 'better', 'right', 'prompt', 'answer', 'wrong', 'response']
Topic 3: ['video', 'use', 'model', 'issues', 'language', 'llms', 'support', 'hey', 'large', 'new']


In [ ]:
# Validation
# Topic Coherence
tokenized_docs = [doc.split() for doc in docs]
dictionary = Dictionary(tokenized_docs)
coh = CoherenceModel(topics=topic_words, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
print(f"Neutral NMF Coherence: {coh.get_coherence()}")

Neutral NMF Coherence: 0.3702418356067084


In [ ]:
# Topic Diversity
unique_words = set()
all_words = []
for words in topic_words:
    all_words.extend(words)
    unique_words.update(words)

diversity = len(unique_words) / len(all_words)
print(f"NMF Topic Diversity: {diversity}")
# Topics appear to be very diverse

NMF Topic Diversity: 0.95


In [ ]:
seeds = [42, 123, 456]
for seed in seeds:
    docs = neg_clean_data_nmf_yt(neutral_df, keys)
    vectorizer = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=2)
    X = vectorizer.fit_transform(docs)

    nmf = NMF(n_components=4, random_state=seed)
    W = nmf.fit_transform(X)
    H = nmf.components_

    vocab = vectorizer.get_feature_names_out()
    print(f"\nSeed {seed}:")
    for k in range(4):
        top_idx = H[k].argsort()[::-1][:5]
        words = [vocab[i] for i in top_idx]
        print(f"  Topic {k}: {words}")
# No randomness in seeding


Seed 42:
  Topic 0: ['people', 'way', 'good', 'things', 'lot']
  Topic 1: ['real', 'person', 'life', 'god', 'time']
  Topic 2: ['questions', 'answers', 'question', 'thoughts', 'better']
  Topic 3: ['video', 'use', 'model', 'issues', 'language']

Seed 123:
  Topic 0: ['people', 'way', 'good', 'things', 'lot']
  Topic 1: ['real', 'person', 'life', 'god', 'time']
  Topic 2: ['questions', 'answers', 'question', 'thoughts', 'better']
  Topic 3: ['video', 'use', 'model', 'issues', 'language']

Seed 456:
  Topic 0: ['people', 'way', 'good', 'things', 'lot']
  Topic 1: ['real', 'person', 'life', 'god', 'time']
  Topic 2: ['questions', 'answers', 'question', 'thoughts', 'better']
  Topic 3: ['video', 'use', 'model', 'issues', 'language']
